# Task 8 - Fine-tune BERT

Gắn classification head lên BERT đã pretrain rồi fine-tune cho từng subtask. Minh hoạ trên tiếng Anh với `bert-base-uncased`; với tiếng Ý/Hà Lan/Ba Tư (các ngôn ngữ khác của Task 8) nên đổi sang một model đa ngữ như `bert-base-multilingual-cased` hoặc `xlm-roberta-base`.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
!git clone -q https://github.com/hoanhh4977/SemEval2027.git
%cd SemEval2027/

In [ ]:
import pandas as pd

data = pd.read_csv("data/task8/train/StereoQueerEval_EN_training.tsv", sep="\t")
data.head()

## Chuẩn bị nhãn cho 3 subtask

In [ ]:
# Subtask A - Stereotype Detection (nhị phân)
label2id_A = {"yes": 0, "no": 1}
id2label_A = {v: k for k, v in label2id_A.items()}
data["label_A"] = data["stereotype"].map(label2id_A)

# Subtask B - Hate Speech Classification (3 lớp)
label2id_B = {"no": 0, "yes_implicit": 1, "yes_explicit": 2}
id2label_B = {v: k for k, v in label2id_B.items()}
data["label_B"] = data["hate_speech"].map(label2id_B)

# Subtask C - Target & Identity Identification (multilabel, chỉ trên comment hateful)
def parse_target(x):
    if x == "none":
        return []
    scope, identities = x.split("_", 1)
    return [scope] + identities.split(",")

data["target_labels"] = data["target"].apply(parse_target)
data_c = data[data["hate_speech"] != "no"].reset_index(drop=True)
print(f"Subtask C chỉ dùng {len(data_c)}/{len(data)} comment hateful")

## Chia Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

train_A, test_A = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_A"])
train_B, test_B = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_B"])
train_C, test_C = train_test_split(data_c, test_size=0.2, random_state=42)

print(f"A: {len(train_A)}/{len(test_A)}  B: {len(train_B)}/{len(test_B)}  C: {len(train_C)}/{len(test_C)}")

## Tokenizer dùng chung

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess(batch):
    return tokenizer(batch["yt_comment"], truncation=True)

## Hàm huấn luyện dùng chung cho cả 3 subtask

Chỉ cần đổi `num_labels` và `problem_type` ("single_label_classification" cho nhị phân/đa lớp, "multi_label_classification" cho multilabel) là dùng lại được cho mọi subtask, không cần viết lại vòng lặp huấn luyện 3 lần.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def build_dataset(train_df, test_df, label_col):
    dd = DatasetDict({
        "train": Dataset.from_pandas(
            train_df[["yt_comment", label_col]].rename(columns={label_col: "label"}),
            preserve_index=False,
        ),
        "test": Dataset.from_pandas(
            test_df[["yt_comment", label_col]].rename(columns={label_col: "label"}),
            preserve_index=False,
        ),
    })
    return dd.map(preprocess, remove_columns=["yt_comment"])

def compute_metrics_single(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

def train_bert(train_df, test_df, label_col, num_labels,
                problem_type="single_label_classification",
                id2label=None, label2id=None):
    dataset_dict = build_dataset(train_df, test_df, label_col)
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=num_labels, problem_type=problem_type,
        id2label=id2label, label2id=label2id,
    )
    args = TrainingArguments(
        output_dir="bert_run", learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=2, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="no", report_to="none",
    )
    compute_fn = (
        compute_metrics_multilabel
        if problem_type == "multi_label_classification"
        else compute_metrics_single
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=dataset_dict["train"], eval_dataset=dataset_dict["test"],
        processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_fn,
    )
    trainer.train()
    return trainer, trainer.evaluate()

## Subtask A: Stereotype Detection

In [ ]:
trainer_A, metrics_A = train_bert(
    train_A, test_A, "label_A", num_labels=2,
    id2label=id2label_A, label2id=label2id_A,
)
metrics_A

## Subtask B: Hate Speech Classification

In [ ]:
trainer_B, metrics_B = train_bert(
    train_B, test_B, "label_B", num_labels=3,
    id2label=id2label_B, label2id=label2id_B,
)
metrics_B

## Subtask C: Target & Identity Identification (multilabel)

Cần đưa `target_labels` (danh sách nhãn) về dạng vector multi-hot trước khi đưa vào hàm `train_bert`, vì HuggingFace cần nhãn là số, không phải chuỗi.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y_train_C = mlb.fit_transform(train_C["target_labels"]).astype(float)
y_test_C = mlb.transform(test_C["target_labels"]).astype(float)

train_C = train_C.copy()
test_C = test_C.copy()
train_C["label_multihot"] = list(y_train_C)
test_C["label_multihot"] = list(y_test_C)
print("Các nhãn:", list(mlb.classes_))

In [ ]:
trainer_C, metrics_C = train_bert(
    train_C, test_C, "label_multihot",
    num_labels=len(mlb.classes_), problem_type="multi_label_classification",
)
metrics_C